# Modelo de Classificação: Predição de Popularidade Literária
Este notebook documenta o desenvolvimento de um algoritmo de Aprendizado de Máquina Supervisionado cujo objetivo é classificar o potencial de popularidade de obras literárias com base em seus metadados estruturais.

## 1. Carregamento e Inspeção Inicial dos dados
Nesta etapa, realizamos a importação dos conjuntos de dados brutos e pré-processados para avaliar sua dimensionalidade e identificar a estrutura mais adequada para a modelagem preditiva

In [16]:
import pandas as pd

# Importação dos conjuntos de dados
df_books_clean = pd.read_parquet('../data/processed/books_clean.parquet')
df_books = pd.read_parquet('../../datasets/books.parquet')
df_data = pd.read_parquet('../../datasets/data.parquet')

# Análise de dimensionalidade
print(f"Dataset processado (books_clean): {df_books_clean.shape[0]} linhas/instâncias e {df_books_clean.shape[1]} colunas/atributos")
print(f"Dataset bruto (books): {df_books.shape[0]} linhas/instâncias e {df_books.shape[1]} colunas/atributos")
print(f"Dataset bruto (data): {df_data.shape[0]} linhas/instâncias e {df_data.shape[1]} colunas/atributos")

Dataset processado (books_clean): 84054 linhas/instâncias e 14 colunas/atributos
Dataset bruto (books): 84054 linhas/instâncias e 13 colunas/atributos
Dataset bruto (data): 100000 linhas/instâncias e 13 colunas/atributos


In [17]:
print("--- Colunas de books_clean ---")
print(df_books_clean.columns.tolist())

print("\n--- Colunas de books ---")
print(df_books.columns.tolist())

print("\n--- Colunas de data ---")
print(df_data.columns.tolist())

--- Colunas de books_clean ---
['author', 'bookformat', 'desc', 'genre', 'img', 'isbn', 'isbn13', 'link', 'pages', 'rating', 'reviews', 'title', 'totalratings', 'genre_list']

--- Colunas de books ---
['author', 'bookformat', 'desc', 'genre', 'img', 'isbn', 'isbn13', 'link', 'pages', 'rating', 'reviews', 'title', 'totalratings']

--- Colunas de data ---
['author', 'bookformat', 'desc', 'genre', 'img', 'isbn', 'isbn13', 'link', 'pages', 'rating', 'reviews', 'title', 'totalratings']


In [18]:
# Mostra as primeiras linhas do arquivo 'books'
df_books.head(3)

,author,bookformat,desc,genre,img,isbn,isbn13,link,pages,rating,reviews,title,totalratings
0,Laurence M. Hauptman,Hardcover,Reveals that several hundred thousand Indians ...,"History,Military History,Civil War,American Hi...",https://i.gr-assets.com/images/S/compressed.ph...,002914180X,9.78E+12,https://goodreads.com/book/show/1001053.Betwee...,0,3.52,5,Between Two Fires: American Indians in the Civ...,33
1,"Charlotte Fiell,Emmanuelle Dirix",Paperback,Fashion Sourcebook - 1920s is the first book i...,"Couture,Fashion,Historical,Art,Nonfiction",https://i.gr-assets.com/images/S/compressed.ph...,1906863482,9.78E+12,https://goodreads.com/book/show/10010552-fashi...,576,4.51,6,Fashion Sourcebook 1920s,41
2,Andy Anderson,Paperback,The seminal history and analysis of the Hungar...,"Politics,History",https://i.gr-assets.com/images/S/compressed.ph...,948984147,9.78E+12,https://goodreads.com/book/show/1001077.Hungar...,124,4.15,2,Hungary 56,26


## 2. Engenharia de Atributos, Multi-classe e Balanceamento (Undersampling)
Para alinhar o modelo com os critérios de avaliação e mitigar o efeito da cauda longa, estruturamos o problema em 3 classes de popularidade baseadas no volume de avaliações (`totalratings`). 

Como o dataset é altamente desbalanceado (com vasta maioria na Classe 3), aplicaremos a técnica de **Undersampling** utilizando o `RandomUnderSampler` para equilibrar a distribuição das classes antes de alimentar o algoritmo KNN, prevenindo o viés algorítmico.  

Nesta etapa, implementamos:
1. **Engenharia de Frequência para Alta Cardinalidade:** Mensuração da relevância do autor com base no seu histórico de publicações no dataset.
2. **One-Hot Encoding para o Formato:** Agrupamento de formatos raros e codificação binária dos formatos majoritários (`bookformat`).
3. **Matriz Esparsa e TruncatedSVD para Gêneros:** Desnormalização dos múltiplos rótulos de gêneros literários de forma computacionalmente eficiente.

In [19]:
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.decomposition import TruncatedSVD

# 1. Higienização base
df_ml = df_books_clean[df_books_clean['pages'] > 0].copy()
df_ml = df_ml.dropna(subset=['pages', 'bookformat', 'genre', 'author', 'totalratings'])

# 2. Definição das três classes
def definir_tres_classes(ratings):
    if ratings >= 10000:
        return 1  # Classe 1: Bestseller
    elif ratings >= 1000:
        return 2  # Classe 2: Média Popularidade
    else:
        return 3  # Classe 3: Nicho / Baixa Popularidade (Maioria)
 
df_ml['popularity_class'] = df_ml['totalratings'].apply(definir_tres_classes)

# 3. Definição das métricas utilizadas

# 3.1 Relevância de cada autor
# Contamos quantos livros cada autor possui no banco para medir sua "importância" 
autor_frequencia = df_ml['author'].value_counts()
df_ml['author_frequency'] = df_ml['author'].map(autor_frequencia)

# 3.2 Formato do livro (one-hot encoding)
# Identificamos os 5 formatos mais comuns e agrupamos o restante como "Other"
top_formatos = df_ml['bookformat'].value_counts().index[:5]
df_ml['format_grouped'] = df_ml['bookformat'].apply(lambda x: x if x in top_formatos else 'Other')
df_formatos_encoded = pd.get_dummies(df_ml['format_grouped'], prefix='format', drop_first=True)

# 3.3 Gêneros (Matriz Esparsa + SVD)
# Transformação da string em lista de gêneros
df_ml['lista_generos'] = df_ml['genre'].apply(lambda x: [g.strip() for g in x.split(',') if g.strip()])
mlb = MultiLabelBinarizer(sparse_output=True)
matriz_esparsa_generos = mlb.fit_transform(df_ml['lista_generos'])

# 3.5 Redução de Dimensionalidade (TruncatedSVD) para evitar a Maldição da Dimensionalidade
svd_generos = TruncatedSVD(n_components=10, random_state=42)
generos_comprimidos = svd_generos.fit_transform(matriz_esparsa_generos)
df_generos_svd = pd.DataFrame(generos_comprimidos, columns=[f'gen_svd_{i}' for i in range(1, 11)], index=df_ml.index)

# 4. Consolidação dos Preditores X e Y
X_estrutural = pd.concat([df_ml[['pages', 'author_frequency']], df_formatos_encoded, df_generos_svd], axis=1)
y_multi = df_ml['popularity_class']

y_estrutural = df_ml['popularity_class']

print("Distribuição das 3 classes no catálogo:")
print(y_multi.value_counts())

Distribuição das 3 classes no catálogo:
popularity_class
3    58949
2    16921
1     3884
Name: count, dtype: int64


## 3. Divisão dos Dados, Padronização e Treinamento do KNN
Com o conjunto de dados estruturais realizamos a divisão em treino/teste e aplicamos a padronização matemática (`StandardScaler`), crucial para o funcionamento correto do cálculo de distâncias do KNN.

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score
from imblearn.under_sampling import RandomUnderSampler

# 1. Divisão Hold-Out (70% treino/ 30% teste)
x_train, x_teste, y_train, y_teste = train_test_split(X_estrutural, y_multi, test_size=0.3, random_state=42, stratify=y_multi)

# 2. Padronização dos dados / z-score
scaler_estrutural = StandardScaler()
x_train_scaled = scaler_estrutural.fit_transform(x_train)
x_teste_scaled = scaler_estrutural.transform(x_teste)

# 3. Aplicação do Balanceamento via Undersampling (apenas no conjunto de treino)
print("\nAplicando Undersampling no conjunto de treino...")
rus = RandomUnderSampler(random_state=42)
X_train_resampled, y_train_resampled = rus.fit_resample(x_train_scaled, y_train)

print("Distribuição balanceada das classes para o treino do KNN:")
print(pd.Series(y_train_resampled).value_counts())

# 4. Predição e avaliação
print("Treinando o modelo KNN + UNDERSMPL...")
modelo_knn = KNeighborsClassifier(n_neighbors=5)
modelo_knn.fit(X_train_resampled, y_train_resampled)

# 5. Predição no conjunto de teste
previsoes = modelo_knn.predict(x_teste_scaled)


Aplicando Undersampling no conjunto de treino...
Distribuição balanceada das classes para o treino do KNN:
popularity_class
1    2719
2    2719
3    2719
Name: count, dtype: int64
Treinando o modelo KNN + UNDERSMPL...


In [21]:
# 6. Exibição do relatório formatado igual à tabela modelo
print("=-" * 35)
print("             TABELA DE RESULTADOS DO TREINO (KNN + UNDERSMPL)          ")
print("=" * 70)
print(f"Acurácia Global do Modelo: {accuracy_score(y_teste, previsoes) * 100:.2f}%\n")
print(classification_report(y_teste, previsoes, target_names=['Classe 1', 'Classe 2', 'Classe 3']))
print("-=" * 35)

=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
             TABELA DE RESULTADOS DO TREINO (KNN + UNDERSMPL)          
Acurácia Global do Modelo: 67.74%

              precision    recall  f1-score   support

    Classe 1       0.18      0.66      0.29      1165
    Classe 2       0.41      0.48      0.44      5077
    Classe 3       0.95      0.73      0.83     17685

    accuracy                           0.68     23927
   macro avg       0.51      0.63      0.52     23927
weighted avg       0.80      0.68      0.72     23927

-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=


## Otimização científica do KNN

In [22]:
from sklearn.model_selection import GridSearchCV

# 1. Justificativa Matemática do Limite de K
# Usamos a raiz quadrada do tamanho do treino como limite superior estatístico
n_amostras_treino = X_train_resampled.shape[0]
limite_superior_k = int(np.sqrt(n_amostras_treino))

print(f"Número de amostras no treino balanceado: {n_amostras_treino}")
print(f"Limite matemático sugerido para K (Raiz Quadrada): {limite_superior_k}")

# 2. Geração Programática dos Valores de K (Apenas Ímpares para evitar empates)
# Criamos uma faixa que vai de 3 até o limite (ou até 41 para manter o custo computacional saudável)
valores_k_funcao = [k for k in range(3, min(limite_superior_k, 41), 2)]
print(f"Valores de K determinados por função para teste: {valores_k_funcao}\n")

# 3. Configuração do Grid de Parâmetros
param_grid = {
    'n_neighbors': valores_k_funcao,
    'weights': ['uniform', 'distance']
}

# 4. Instanciando o Buscador com Validação Cruzada (Stratified 5-Fold)
# O 'scoring' f1_macro garante que ele busque o melhor equilíbrio entre as 3 classes
knn_base = KNeighborsClassifier()
grid_search_knn = GridSearchCV(
    estimator=knn_base, 
    param_grid=param_grid, 
    scoring='f1_macro', 
    cv=5, 
    verbose=1, 
    n_jobs=-1
)

# 5. Executando a Busca Científica
print("Iniciando GridSearchCV com Validação Cruzada de 5 dobras...")
grid_search_knn.fit(X_train_resampled, y_train_resampled)

# 6. Extraindo as Melhores Configurações Justificadas
print("\n=======================================================================")
print("                      MELHOR CONFIGURAÇÃO ENCONTRADA                    ")
print("=======================================================================")
print(f"Melhor valor de K encontrado: {grid_search_knn.best_params_['n_neighbors']}")
print(f"Melhor tipo de peso encontrado: '{grid_search_knn.best_params_['weights']}'")
print(f"F1-Score Médio de Validação: {grid_search_knn.best_score_ * 100:.2f}%")
print("=======================================================================")

# Avaliação final no conjunto de teste usando o melhor modelo gerado automaticamente
melhor_modelo_knn = grid_search_knn.best_estimator_
previsoes_otimizadas = melhor_modelo_knn.predict(x_teste_scaled)

print("\nRelatório de Classificação Final do Modelo Otimizado:")
print(classification_report(y_teste, previsoes_otimizadas, target_names=['Classe 1', 'Classe 2', 'Classe 3']))

Número de amostras no treino balanceado: 8157
Limite matemático sugerido para K (Raiz Quadrada): 90
Valores de K determinados por função para teste: [3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39]

Iniciando GridSearchCV com Validação Cruzada de 5 dobras...
Fitting 5 folds for each of 38 candidates, totalling 190 fits

                      MELHOR CONFIGURAÇÃO ENCONTRADA                    
Melhor valor de K encontrado: 15
Melhor tipo de peso encontrado: 'distance'
F1-Score Médio de Validação: 64.59%

Relatório de Classificação Final do Modelo Otimizado:
              precision    recall  f1-score   support

    Classe 1       0.23      0.65      0.34      1165
    Classe 2       0.43      0.51      0.47      5077
    Classe 3       0.93      0.77      0.85     17685

    accuracy                           0.71     23927
   macro avg       0.53      0.64      0.55     23927
weighted avg       0.79      0.71      0.74     23927



## 4. Persistência dos Componentes de Produção
Para que o aplicativo funcione corretamente, precisamos salvar não apenas o modelo e o scaler, mas também os transformadores categóricos (`MultiLabelBinarizer` e `TruncatedSVD`) utilizados nos gêneros.

In [23]:
import joblib
import os

# Definição de caminhos para a pasta de modelo do app
models_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "Machine Learning", "models"))
os.makedirs(models_dir, exist_ok=True)

# Salvando todos os componentes necessários para o back-end
joblib.dump(modelo_knn, os.path.join(models_dir, "knn_popularidade.pkl"))
joblib.dump(scaler_estrutural, os.path.join(models_dir, "scaler_popularidade.pkl"))
joblib.dump(mlb, os.path.join(models_dir, "mlb_generos.pkl"))
joblib.dump(svd_generos, os.path.join(models_dir, "svd_generos.pkl"))
joblib.dump(autor_frequencia, os.path.join(models_dir, "autor_frequencia.pkl")) # Para o app mapear o autor

print("Todos os novos artefatos (.pkl) foram exportados para a produção.")


Todos os novos artefatos (.pkl) foram exportados para a produção.
